In [1]:
# for reading env file
from dotenv import load_dotenv
import os

# for loading db
import pymysql

# for working part
import numpy as np
import pandas as pd

# for dropdowns selection for user
import ipywidgets as widgets
from IPython.display import display

# for config file 
import json

In [15]:
## reading my sql database through env

load_dotenv()



conn = pymysql.connect(
    host=os.getenv("host"),
    port=int(os.getenv("port")),
    user=os.getenv("user"),
    password=os.getenv("password"),
    database=os.getenv("database"),
    ssl={"ssl_mode": "REQUIRED"}
)

cursor = conn.cursor()



print("the db is connected database name is ",os.getenv("database"))

the db is connected database name is  pharma_db


In [3]:
# visibility of all table names

table_list = pd.read_sql("show tables;",conn)
table_list = list(table_list.iloc[:,0])


print("table names in db please select the table names that are required in next step: ", table_list)



C:\Users\hp5cd\AppData\Local\Temp\ipykernel_6432\3884484491.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  table_list = pd.read_sql("show tables;",conn)


table names in db please select the table names that are required in next step:  ['agents', 'bulk', 'bulk_compliance', 'bulk_technical', 'calls', 'contacts', 'leads', 'order_history', 'order_items', 'orders', 'packaging_cost', 'packaging_master', 'packaging_supplier', 'pii', 'product_channel', 'product_cost', 'product_description', 'product_images', 'product_packaging', 'products', 'raw_technicals', 'supplier', 'supplier_technicals', 'technical_compliance', 'technicals']


In [7]:
# asking user which tables in the database are useful for this project

def select_tables():
    selector = widgets.SelectMultiple(options=table_list,description='select table cltrl+click',rows=(min(10,len(table_list))))

    display(selector)

    return selector



In [8]:

table_selector = select_tables()

SelectMultiple(description='select table cltrl+click', options=('agents', 'bulk', 'bulk_compliance', 'bulk_tec…

In [9]:


selected_tables = list(table_selector.value)

print(selected_tables)

['calls', 'contacts', 'leads', 'orders']


In [11]:


config = {
    "selected_tables": selected_tables
}

with open("../config/selected_tables.json", "w") as f:
    json.dump(config, f, indent=4)

In [17]:
def read_db(tn, limit=50):

    tables = {}
    columns_dict = {}

    for table in tn:

        df = pd.read_sql(
            f"SELECT * FROM {table} LIMIT {limit};",
            conn
        )

        tables[f"{table}_df"] = df

        columns_dict[table] = df.columns.tolist()

        print(f"\n{'='*60}")
        print(f"TABLE: {table}")
        print(df.columns)
        print("column count", df.shape[1])

    return tables, columns_dict

In [21]:
tables, columns_dict = read_db(selected_tables,100)

C:\Users\hp5cd\AppData\Local\Temp\ipykernel_6432\255727841.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(



TABLE: calls
Index(['id', 'call_id', 'pii_id', 'call_date', 'duration', 'outcome',
       'call_type', 'recording', 'agent_id'],
      dtype='object')
column count 9

TABLE: contacts
Index(['id', 'contact_id', 'lead_id', 'pii_id', 'profile', 'profile_details',
       'current_pipeline_stage'],
      dtype='object')
column count 7

TABLE: leads
Index(['id', 'lead_id', 'pii_id', 'name', 'owner', 'current_pipeline_stage',
       'lead_source', 'assigned_date', 'lead_quality_score', 'follow_up_date'],
      dtype='object')
column count 10

TABLE: orders
Index(['id', 'order_id', 'pii_id', 'payment_mode', 'courier_name',
       'tracking_link', 'awb_number', 'order_amount', 'freight_charges',
       'agent_id', 'channel', 'order_date', 'order_status', 'exchange_rate'],
      dtype='object')
column count 14


In [ ]:
# all tables can be connected through pii_id [FK]
# lead_id in leads table, contact_id in contacts table, call_id in calls table, and order_id in orders table [PK]


In [19]:
# user input for which columns to consider for the analysis and prediction

def select_columns(columns_dict):

    selectors = {}

    for table, columns in columns_dict.items():

        selector = widgets.SelectMultiple(
            options=columns,
            description=table,
            rows=min(15, len(columns))
        )

        display(selector)

        selectors[table] = selector

    return selectors

In [22]:
column_selectors = select_columns(columns_dict)

SelectMultiple(description='calls', options=('id', 'call_id', 'pii_id', 'call_date', 'duration', 'outcome', 'c…

SelectMultiple(description='contacts', options=('id', 'contact_id', 'lead_id', 'pii_id', 'profile', 'profile_d…

SelectMultiple(description='leads', options=('id', 'lead_id', 'pii_id', 'name', 'owner', 'current_pipeline_sta…

SelectMultiple(description='orders', options=('id', 'order_id', 'pii_id', 'payment_mode', 'courier_name', 'tra…

In [23]:
selected_columns = {
    table: list(widget.value)
    for table, widget in column_selectors.items()
}


with open("../config/selected_columns.json", "w") as f:
    json.dump(
        selected_columns,
        f,
        indent=4
    )
